In [1]:
import os
import argparse
import torch
from datasets import MyDataSet
# from vit_model import VisionTransformer
from Residual import Residual
from Residual import Student


import collections
import math
import shutil
import pandas as pd
import numpy as np
import torchvision
from torch import nn
from torch.utils.data import Dataset
from torch.nn import functional as F
from d2l import torch as d2l
from PIL import Image

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
singlefile_data_num = 400  #每个文件训练读取数据个数
singlefile_val_data_num = 100 #每个文件验证读取数据个数
batch_size = 32
epochs = 101
lr = 0.005    #学习率大小
train_path = r"/home/luhan/lap/IndooLocation/Data/srs-38points-1000time-1cycles/01"
test_path =  r"/home/luhan/lap/IndooLocation/Data/srs-38points-1000time-1cycles/01"
if os.path.exists("./weights") is False:
    os.makedirs("./weights")

In [3]:
# 实例化训练数据集
train_dataset = MyDataSet(folder_path=train_path)

valid_len is 38912000
origin_len is 38912000
输入数据总维度为  (38000, 1, 1024, 8)
标签总维度为  (38000, 1, 1024)


In [4]:
# 实例化测试数据集
val_dataset = MyDataSet(folder_path = test_path)

valid_len is 38912000
origin_len is 38912000
输入数据总维度为  (38000, 1, 1024, 8)
标签总维度为  (38000, 1, 1024)


In [5]:
nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])  # number of workers
nw = 0 # in windows
print('Using {} dataloader workers every process'.format(nw))

train_loader = torch.utils.data.DataLoader(train_dataset,
                                            batch_size=batch_size,  #weight_decay=1e-3
                                            shuffle=True,
                                            pin_memory=True,
                                            num_workers=nw)

val_loader = torch.utils.data.DataLoader(val_dataset,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            pin_memory=True,
                                            num_workers=nw)
                                            # 清空txt数据

Using 0 dataloader workers every process


In [6]:
with open("loss.txt", "w") as f:
    f.write("")
with open("accuracy.txt", "w") as f:
    f.write("")
with open("accuracy_test.txt", "w") as f:
    f.write("")
with open("loss_test.txt", "w") as f:
    f.write("")
with open("train_predictions.csv", "w") as f:
    f.write("")

In [7]:
print("total epochs:",epochs)
from utils import train_one_epoch,test_model
print("using ",torch.cuda.is_available())
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
model=Student().to(device)
model.train()
optimizer = torch.optim.SGD(model.parameters(), lr = lr)
for epoch in range(epochs):
    # train
    train_loss, train_accuracy = train_one_epoch(model=model,
                                optimizer=optimizer,
                                data_loader=train_loader,
                                device=device,
                                epoch=epoch,
                                save_path="train_predictions.csv")
    optimizer.step()
    with open("loss.txt", "a") as f_loss:
        f_loss.write("loss:{}\n".format(train_loss))
    with open("accuracy.txt", "a") as f_accuracy:
        f_accuracy.write("accuracy:{}\n".format(train_accuracy))
    # validate
    # if (epoch+1) % 5 == 0:
    #     pred, test_loss, labels, test_accuracy = test_model(model=model,
    #                             data_loader=val_loader,
    #                             device=device)
    #     average_loss = sum(test_loss)/len(test_loss)
    #     print("................")
    #     print("验证集结果：")
    #     print(f"平均误差: {average_loss:.3f}")
    #     print(f"准确率: {test_accuracy:.2f}%")
    #     print("................")
    #     with open("accuracy_test.txt", "a") as f:
    #         f.write("accuracy test:{}\n".format(test_accuracy))
    #     with open("loss_test.txt", "a") as f:
    #         f.write("loss test:{}\n".format(average_loss))
        
    if (epoch+1) % 20 == 0:
        print("保存模型")
        # torch.save(model.state_dict(), "./weights/model-{}.pth".format(epoch))
        if not os.path.exists('./model_save'):
            os.makedirs('./model_save')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': train_loss,
            'accuracy': train_accuracy
            }, "./model_save/model-{}.pth".format(epoch))
print("训练完成")

total epochs: 101
using  True
 
[train epoch 1] 平均误差:0.006, 平均准确率:0.00%
 
[train epoch 2] 平均误差:0.001, 平均准确率:0.00%
 
[train epoch 3] 平均误差:0.001, 平均准确率:0.00%
 
[train epoch 4] 平均误差:0.001, 平均准确率:0.00%
 
[train epoch 5] 平均误差:0.001, 平均准确率:0.00%


KeyboardInterrupt: 

In [ ]:
# pred, test_loss, labels, test_accuracy = test_model(model=model,
#                                 data_loader=val_loader,
#                                 device=device)
# average_loss = sum(test_loss)/len(test_loss)
# print("................")
# print("验证集结果：")
# print(f"平均误差: {average_loss:.3f}")
# print(f"准确率: {test_accuracy:.2f}%")
# print("................")

In [ ]:
# from torchinfo import summary

# model = Student()
# summary(model, input_size=(1, 1, 256, 8), col_names=["input_size", "output_size", "num_params", "kernel_size"])